## Global optimized landmarks over keyframes (from logged `meta_data.npz`)

This notebook:
- Loads `meta_data.npz` written by the Point2Pose pipeline (`save_meta_data: true`)
- Finds keyframes (`is_key_frame == True`)
- Replays a lightweight global landmark optimizer (LM graph) over keyframes to recover
  `key_points_optimized` + `key_points_idx_optimized` per keyframe update
- Plots an interactive 3D Plotly slider similar to `12a. frame_optimization_analysis_debug.ipynb`

### Usage

1. Set `META_DATA_NPZ` in the next cell.
2. Run all cells.


In [ ]:
import os
import sys
import numpy as np

# --- User config ---
# Path to the logged NPZ (DataLogger output)
META_DATA_NPZ = "/home/justin/code/point-to-pose/results/ho3d_single/MPM12/meta_data/meta_data.npz"

OBJ_ID = 0
MAX_KEYFRAMES = None  # e.g. 40 to limit runtime

# Plot params
PLOT_WIDTH = 1200
PLOT_HEIGHT = 850
FIX_AXIS_RANGES = True
ASPECT_MODE = "cube"  # "data" or "cube"

# Replay params
# POSE_FIELD_PREFERENCE = ("pose_local", "pose_frontend", "obj_pose")
POSE_FIELD_PREFERENCE="obj_pose"
SUPPRESS_OPTIMIZER_PRINTS = True

# Ensure repo root is importable (when running from the `notebook/` directory)
_cwd = os.path.abspath(os.getcwd())
_repo_root = os.path.abspath(os.path.join(_cwd, "..")) if os.path.basename(_cwd) == "notebook" else _cwd
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

assert os.path.exists(META_DATA_NPZ), f"META_DATA_NPZ not found: {META_DATA_NPZ}"


In [ ]:
import contextlib
import io


def _ragged_slice(npz, key: str, row_idx: int) -> np.ndarray:
    """Return the flattened ragged payload for one row."""
    data = npz[f"{key}_data"]
    offsets = npz[f"{key}_offsets"]
    lengths = npz[f"{key}_lengths"]

    off = int(offsets[row_idx])
    L = int(lengths[row_idx])
    if L <= 0:
        return np.asarray([], dtype=data.dtype)
    return np.asarray(data[off : off + L])


def _ragged_slice_2d(npz, key: str, row_idx: int, d: int, *, dtype=None) -> np.ndarray:
    """Return ragged payload reshaped as (-1, d)."""
    flat = _ragged_slice(npz, key, row_idx)
    if dtype is not None:
        flat = flat.astype(dtype, copy=False)
    if flat.size == 0:
        return np.zeros((0, d), dtype=(dtype or flat.dtype))
    if flat.size % d != 0:
        raise ValueError(f"Ragged field {key} row {row_idx}: flat.size={flat.size} not divisible by d={d}")
    return flat.reshape(-1, d)


def _choose_pose(meta, row_idx: int, prefer=POSE_FIELD_PREFERENCE) -> np.ndarray:
    """Pick a 4x4 pose matrix from the logged meta arrays."""
    for k in prefer:
        if k not in meta.files:
            continue
        p = meta[k][row_idx]
        if p is None:
            continue
        p = np.asarray(p, dtype=float)
        if p.shape == (4, 4):
            return p
    raise KeyError(f"None of pose fields present/valid at row {row_idx}: {prefer}")


In [ ]:
# Load logged meta data
meta = np.load(META_DATA_NPZ, allow_pickle=True)

if "frame_id" not in meta.files or "is_key_frame" not in meta.files:
    raise KeyError(
        "meta_data.npz is missing required keys. Expected at least: frame_id, is_key_frame. "
        f"Found keys: {sorted(meta.files)[:20]} ..."
    )

frame_ids = np.asarray(meta["frame_id"], dtype=int)
is_key_frame = np.asarray(meta["is_key_frame"], dtype=bool)

kf_rows = np.where(is_key_frame)[0].astype(int)
if MAX_KEYFRAMES is not None:
    kf_rows = kf_rows[: int(MAX_KEYFRAMES)]

print(f"Loaded meta rows: {len(frame_ids)}")
print(f"Keyframes in log:  {int(is_key_frame.sum())} (using {len(kf_rows)})")

# --- Build landmark snapshots directly from the logged pipeline state ---
#
# The pipeline logs `obj_key_points` *after* its per-frame work, including any
# keyframe graph updates. For frames where `is_key_frame == True`, we treat the
# logged `obj_key_points` as the "global optimized landmarks" snapshot.
#
# Landmark IDs here are the (stable) indices in the object's keypoint array.

if "obj_key_points_data" not in meta.files:
    raise KeyError(
        "meta_data.npz is missing obj_key_points (expected ragged fields: obj_key_points_data/offsets/lengths)."
    )

global_landmarks_updates = []

for kf_idx, row_idx in enumerate(kf_rows.tolist()):
    frame_id = int(frame_ids[row_idx])

    xyz = _ragged_slice_2d(meta, "obj_key_points", row_idx, 3, dtype=float)

    # Stable per-landmark ids: index in the object keypoint array
    ids = np.arange(xyz.shape[0], dtype=int)

    # Optionally filter invalid keypoints (keep original indices as ids)
    if "obj_valid_data" in meta.files:
        valid = _ragged_slice(meta, "obj_valid", row_idx).astype(bool, copy=False).reshape(-1)
        if valid.size == xyz.shape[0]:
            ids = np.flatnonzero(valid).astype(int)
            xyz = xyz[valid]

    m = np.isfinite(xyz).all(axis=1)
    xyz = xyz[m]
    ids = ids[m]

    global_landmarks_updates.append(
        {
            "obj_id": int(OBJ_ID),
            "kf_idx": int(kf_idx),
            "frame_id": int(frame_id),
            "xyz": xyz,
            "ids": ids,
        }
    )

print(f"Collected global_landmarks_updates: {len(global_landmarks_updates)}")
if len(global_landmarks_updates) > 0:
    print(
        "First/last kf_idx:",
        global_landmarks_updates[0]["kf_idx"],
        global_landmarks_updates[-1]["kf_idx"],
    )

In [ ]:
# --- Plotly 3D slider (same idea as 12a) ---
if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Run the replay cell above first.")
elif len(global_landmarks_updates) == 0:
    print("global_landmarks_updates is empty. (Likely only the first keyframe ran, or the optimizer returned no landmarks.)")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    if len(updates) == 0:
        print(f"No landmark snapshots found for OBJ_ID={OBJ_ID}.")
    else:
        # Sort by keyframe index to make the slider monotonic
        updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

        try:
            import plotly.graph_objects as go
        except Exception as e:
            raise ImportError(
                "Plotly is required for the interactive 3D plot. Install with `pip install plotly` and re-run this cell."
            ) from e

        # Establish global color range over ids
        _all_ids = []
        _all_xyz = []
        for u in updates:
            ids = u.get("ids", None)
            xyz = u.get("xyz", None)
            if ids is not None:
                ids = np.asarray(ids, dtype=int).reshape(-1)
                if ids.size > 0:
                    _all_ids.append(ids)
            if xyz is not None:
                xyz = np.asarray(xyz, dtype=float)
                m = np.isfinite(xyz).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz[m])

        if len(_all_ids) == 0:
            raise RuntimeError("No landmark ids found in global_landmarks_updates.")
        if len(_all_xyz) == 0:
            raise RuntimeError("No finite landmark xyz found in global_landmarks_updates.")

        all_ids = np.concatenate(_all_ids, axis=0)
        cmin = float(np.min(all_ids))
        cmax = float(np.max(all_ids))

        all_xyz = np.concatenate(_all_xyz, axis=0)
        xmin, ymin, zmin = np.min(all_xyz, axis=0)
        xmax, ymax, zmax = np.max(all_xyz, axis=0)

        # Fixed axis ranges (so switching frames won't auto-rescale)
        scene_axes = dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            aspectmode=ASPECT_MODE,
        )
        if FIX_AXIS_RANGES:
            # Pad and optionally make cubic
            spans = np.array([xmax - xmin, ymax - ymin, zmax - zmin], dtype=float)
            spans = np.maximum(spans, 1e-6)
            pad = 0.05 * float(np.max(spans))

            cx, cy, cz = float(0.5 * (xmin + xmax)), float(0.5 * (ymin + ymax)), float(0.5 * (zmin + zmax))
            if ASPECT_MODE == "cube":
                half = 0.5 * float(np.max(spans)) + pad
                xr = [cx - half, cx + half]
                yr = [cy - half, cy + half]
                zr = [cz - half, cz + half]
            else:
                xr = [float(xmin - pad), float(xmax + pad)]
                yr = [float(ymin - pad), float(ymax + pad)]
                zr = [float(zmin - pad), float(zmax + pad)]

            scene_axes.update(
                dict(
                    xaxis=dict(range=xr, autorange=False),
                    yaxis=dict(range=yr, autorange=False),
                    zaxis=dict(range=zr, autorange=False),
                )
            )

        # Baseline snapshot (first keyframe update)
        base = updates[0]
        xyz0 = np.asarray(base["xyz"], dtype=float)
        ids0 = np.asarray(base["ids"], dtype=int).reshape(-1)
        m0 = np.isfinite(xyz0).all(axis=1)

        fig = go.Figure()

        fig.add_trace(
            go.Scatter3d(
                x=xyz0[m0, 0],
                y=xyz0[m0, 1],
                z=xyz0[m0, 2],
                mode="markers",
                marker=dict(size=2, color="rgba(140,140,140,0.45)"),
                text=[f"id={int(i)}" for i in ids0[m0]],
                hovertemplate="%{text}<extra></extra>",
                name=f"baseline (kf_idx={int(base.get('kf_idx', -1))}, frame_id={int(base.get('frame_id', -1))})",
            )
        )

        # One trace per update; slider toggles visibility
        for k, u in enumerate(updates):
            xyz = np.asarray(u["xyz"], dtype=float)
            ids = np.asarray(u["ids"], dtype=int).reshape(-1)
            m = np.isfinite(xyz).all(axis=1)
            kf_idx = int(u.get("kf_idx", -1))
            frame_id = int(u.get("frame_id", -1))

            fig.add_trace(
                go.Scatter3d(
                    x=xyz[m, 0],
                    y=xyz[m, 1],
                    z=xyz[m, 2],
                    mode="markers",
                    marker=dict(
                        size=4,
                        color=ids[m].astype(float),
                        colorscale="Turbo",
                        cmin=cmin,
                        cmax=cmax,
                        opacity=0.9,
                        colorbar=dict(title="landmark id"),
                    ),
                    text=[f"id={int(i)}" for i in ids[m]],
                    hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                    name=f"kf_idx={kf_idx}, frame_id={frame_id}",
                    visible=(k == 0),
                )
            )

        steps = []
        for k, u in enumerate(updates):
            vis = [True] + [False] * len(updates)  # baseline always on
            vis[1 + k] = True
            kf_idx = int(u.get("kf_idx", -1))
            frame_id = int(u.get("frame_id", -1))
            steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": vis},
                        {
                            "title": f"Global optimized landmarks over keyframes (obj {OBJ_ID}) — kf_idx={kf_idx}, frame_id={frame_id}",
                        },
                    ],
                    label=str(frame_id if frame_id >= 0 else kf_idx),
                )
            )

        fig.update_layout(
            title=f"Global optimized landmarks over keyframes (obj {OBJ_ID}) — kf_idx={int(updates[0].get('kf_idx', -1))}, frame_id={int(updates[0].get('frame_id', -1))}",
            margin=dict(l=0, r=0, b=0, t=55),
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            scene=scene_axes,
            sliders=[
                dict(
                    active=0,
                    currentvalue={"prefix": "frame_id: "},
                    steps=steps,
                )
            ],
            legend=dict(x=0.01, y=0.99),
            # Keeps user-driven camera/zoom across slider steps
            uirevision=f"kf_landmarks_obj_{OBJ_ID}",
        )

        fig.show()


In [ ]:
# --- Plotly 3D slider with observed points and correspondences ---
if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Run the replay cell above first.")
elif len(global_landmarks_updates) == 0:
    print("global_landmarks_updates is empty. (Likely only the first keyframe ran, or the optimizer returned no landmarks.)")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    if len(updates) == 0:
        print(f"No landmark snapshots found for OBJ_ID={OBJ_ID}.")
    else:
        # Sort by keyframe index to make the slider monotonic
        updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

        try:
            import plotly.graph_objects as go
        except Exception as e:
            raise ImportError(
                "Plotly is required for the interactive 3D plot. Install with `pip install plotly` and re-run this cell."
            ) from e

        # Import transform utilities
        from point2pose.utils.transform import transform_pts, inverse_SE3

        # Load observed points and correspondences for each keyframe
        keyframe_data = []
        for kf_idx, row_idx in enumerate(kf_rows.tolist()):
            frame_id = int(frame_ids[row_idx])
            
            # Get object pose for this keyframe (to transform camera frame to object frame)
            # Follow the same pattern as notebook 12. frame_optimization_analysis.ipynb
            obj_pose = None
            if 'pose_frontend' in meta.files:
                obj_pose = meta['pose_frontend'][row_idx]
            elif 'pose_local' in meta.files:
                obj_pose = meta['pose_local'][row_idx]
            elif 'obj_pose' in meta.files:
                obj_pose = meta['obj_pose'][row_idx]
            elif 'obj_init_pose' in meta.files:
                obj_pose = meta['obj_init_pose'][row_idx]
            
            if obj_pose is not None:
                obj_pose = np.asarray(obj_pose, dtype=float)
                if obj_pose.shape == (4, 4):
                    # obj_pose transforms from object frame to camera frame
                    # We need inverse to transform camera frame to object frame
                    T_c2o = inverse_SE3(obj_pose)
                else:
                    T_c2o = None
            else:
                # If pose not available, skip transformation (will show incorrect positions)
                T_c2o = None
            
            
            # Get global optimized landmarks (map points)
            xyz_map = _ragged_slice_2d(meta, "obj_key_points", row_idx, 3, dtype=float)
            ids_map = np.arange(xyz_map.shape[0], dtype=int)
            
            # Filter valid keypoints
            if "obj_valid_data" in meta.files:
                valid = _ragged_slice(meta, "obj_valid", row_idx).astype(bool, copy=False).reshape(-1)
                if valid.size == xyz_map.shape[0]:
                    ids_map = np.flatnonzero(valid).astype(int)
                    xyz_map = xyz_map[valid]
            
            m = np.isfinite(xyz_map).all(axis=1)
            xyz_map = xyz_map[m]
            ids_map = ids_map[m]
            
            # Get observed points (reg_curr3d) and corresponding map points (reg_key_points)
            # Note: reg_curr3d is in camera frame, reg_key_points is in object frame
            xyz_observed = None
            xyz_correspond = None
            if "reg_curr3d_data" in meta.files and "reg_key_points_data" in meta.files:
                xyz_observed_cam = _ragged_slice_2d(meta, "reg_curr3d", row_idx, 3, dtype=float)
                xyz_correspond = _ragged_slice_2d(meta, "reg_key_points", row_idx, 3, dtype=float)
                
                # Transform observed points from camera frame to object frame
                if T_c2o is not None and xyz_observed_cam.shape[0] > 0:
                    xyz_observed = transform_pts(T_c2o, xyz_observed_cam)
                else:
                    xyz_observed = xyz_observed_cam.copy() if xyz_observed_cam.shape[0] > 0 else np.zeros((0, 3), dtype=float)
                
                # Filter finite points - only if both arrays have the same number of points
                if xyz_observed.shape[0] == xyz_correspond.shape[0] and xyz_observed.shape[0] > 0:
                    m_obs = np.isfinite(xyz_observed).all(axis=1) & np.isfinite(xyz_correspond).all(axis=1)
                    if np.any(m_obs):
                        xyz_observed = xyz_observed[m_obs]
                        xyz_correspond = xyz_correspond[m_obs]
                    else:
                        xyz_observed = np.zeros((0, 3), dtype=float)
                        xyz_correspond = np.zeros((0, 3), dtype=float)
                else:
                    # Mismatched sizes or empty arrays - filter each separately but don't create correspondences
                    if xyz_observed.shape[0] > 0:
                        m_obs = np.isfinite(xyz_observed).all(axis=1)
                        xyz_observed = xyz_observed[m_obs] if np.any(m_obs) else np.zeros((0, 3), dtype=float)
                    else:
                        xyz_observed = np.zeros((0, 3), dtype=float)
                    # Set correspond to empty since sizes don't match
                    xyz_correspond = np.zeros((0, 3), dtype=float)
            
            # Get residuals for correspondence error coloring
            residuals = None
            if "reg_residuals_data" in meta.files and xyz_observed is not None and xyz_correspond is not None:
                residuals_flat = _ragged_slice(meta, "reg_residuals", row_idx)
                if residuals_flat.size > 0:
                    residuals = np.asarray(residuals_flat, dtype=float)
                    # Filter to match the filtered observed/correspond points
                    # We need to apply the same filtering that was applied to observed/correspond points
                    if "reg_curr3d_data" in meta.files and "reg_key_points_data" in meta.files:
                        xyz_observed_cam_raw = _ragged_slice_2d(meta, "reg_curr3d", row_idx, 3, dtype=float)
                        xyz_correspond_raw = _ragged_slice_2d(meta, "reg_key_points", row_idx, 3, dtype=float)
                        
                        if residuals.size == xyz_observed_cam_raw.shape[0] and xyz_observed_cam_raw.shape[0] == xyz_correspond_raw.shape[0]:
                            # Apply same filtering as observed points
                            m_obs = np.isfinite(xyz_observed_cam_raw).all(axis=1) & np.isfinite(xyz_correspond_raw).all(axis=1)
                            if np.any(m_obs):
                                residuals = residuals[m_obs]
                            else:
                                residuals = np.zeros((0,), dtype=float)
                        else:
                            residuals = np.zeros((0,), dtype=float)
                    else:
                        residuals = np.zeros((0,), dtype=float)
                else:
                    residuals = np.zeros((0,), dtype=float)
            else:
                residuals = np.zeros((0,), dtype=float)
            
            keyframe_data.append({
                "kf_idx": int(kf_idx),
                "frame_id": int(frame_id),
                "xyz_map": xyz_map,
                "ids_map": ids_map,
                "xyz_observed": xyz_observed if xyz_observed is not None else np.zeros((0, 3), dtype=float),
                "xyz_correspond": xyz_correspond if xyz_correspond is not None else np.zeros((0, 3), dtype=float),
                "residuals": residuals if residuals is not None else np.zeros((0,), dtype=float),
            })

        # Establish axis ranges (no longer need color range since using fixed colors)
        _all_xyz = []
        for kfd in keyframe_data:
            xyz = kfd.get("xyz_map", None)
            if xyz is not None and xyz.size > 0:
                m = np.isfinite(xyz).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz[m])
            
            # Also include observed points for axis range
            xyz_obs = kfd.get("xyz_observed", None)
            if xyz_obs is not None and xyz_obs.size > 0:
                m = np.isfinite(xyz_obs).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz_obs[m])

        if len(_all_xyz) == 0:
            raise RuntimeError("No finite landmark xyz found in keyframe_data.")
        
        # Error threshold for correspondence coloring (adjust as needed)
        ERROR_THRESHOLD = 0.01  # meters

        all_xyz = np.concatenate(_all_xyz, axis=0)
        xmin, ymin, zmin = np.min(all_xyz, axis=0)
        xmax, ymax, zmax = np.max(all_xyz, axis=0)

        # Fixed axis ranges
        scene_axes = dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            aspectmode=ASPECT_MODE,
        )
        if FIX_AXIS_RANGES:
            spans = np.array([xmax - xmin, ymax - ymin, zmax - zmin], dtype=float)
            spans = np.maximum(spans, 1e-6)
            pad = 0.05 * float(np.max(spans))

            cx, cy, cz = float(0.5 * (xmin + xmax)), float(0.5 * (ymin + ymax)), float(0.5 * (zmin + zmax))
            if ASPECT_MODE == "cube":
                half = 0.5 * float(np.max(spans)) + pad
                xr = [cx - half, cx + half]
                yr = [cy - half, cy + half]
                zr = [cz - half, cz + half]
            else:
                xr = [float(xmin - pad), float(xmax + pad)]
                yr = [float(ymin - pad), float(ymax + pad)]
                zr = [float(zmin - pad), float(zmax + pad)]

            scene_axes.update(
                dict(
                    xaxis=dict(range=xr, autorange=False),
                    yaxis=dict(range=yr, autorange=False),
                    zaxis=dict(range=zr, autorange=False),
                )
            )

        # Baseline snapshot (first keyframe)
        base = keyframe_data[0]
        xyz0 = np.asarray(base["xyz_map"], dtype=float)
        ids0 = np.asarray(base["ids_map"], dtype=int).reshape(-1)
        m0 = np.isfinite(xyz0).all(axis=1)

        fig = go.Figure()

        # Baseline map (always visible) - same pretty blue color
        MAP_COLOR_BASELINE = "rgba(70,130,180,0.45)"  # Steel blue, more transparent for baseline
        fig.add_trace(
            go.Scatter3d(
                x=xyz0[m0, 0],
                y=xyz0[m0, 1],
                z=xyz0[m0, 2],
                mode="markers",
                marker=dict(size=2, color=MAP_COLOR_BASELINE),
                text=[f"id={int(i)}" for i in ids0[m0]],
                hovertemplate="%{text}<extra></extra>",
                name=f"baseline map (kf_idx={int(base.get('kf_idx', -1))}, frame_id={int(base.get('frame_id', -1))})",
            )
        )

        # For each keyframe, add:
        # 1. Map points (key points) for this keyframe
        # 2. Observed points
        # 3. Correspondence lines
        for k, kfd in enumerate(keyframe_data):
            kf_idx = int(kfd.get("kf_idx", -1))
            frame_id = int(kfd.get("frame_id", -1))
            
            # Map points (key points) for this keyframe - pretty blue color
            xyz_map = np.asarray(kfd["xyz_map"], dtype=float)
            ids_map = np.asarray(kfd["ids_map"], dtype=int).reshape(-1)
            m_map = np.isfinite(xyz_map).all(axis=1)
            
            # Pretty blue color for map points
            MAP_COLOR = "rgba(70,130,180,0.85)"  # Steel blue
            
            fig.add_trace(
                go.Scatter3d(
                    x=xyz_map[m_map, 0],
                    y=xyz_map[m_map, 1],
                    z=xyz_map[m_map, 2],
                    mode="markers",
                    marker=dict(
                        size=4,
                        color=MAP_COLOR,
                        opacity=0.9,
                    ),
                    text=[f"id={int(i)}, frame={frame_id}" for i in ids_map[m_map]],
                    hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                    name=f"key points (kf_idx={kf_idx}, frame_id={frame_id})",
                    visible=(k == 0),
                )
            )
            
            # Observed points (already filtered when loaded) - pretty coral/salmon color
            xyz_observed = np.asarray(kfd["xyz_observed"], dtype=float)
            
            # Pretty coral/salmon color for observed points
            OBSERVED_COLOR = "rgba(255,127,80,0.85)"  # Coral
            
            if xyz_observed.size > 0:
                fig.add_trace(
                    go.Scatter3d(
                        x=xyz_observed[:, 0],
                        y=xyz_observed[:, 1],
                        z=xyz_observed[:, 2],
                        mode="markers",
                        marker=dict(
                            size=5,
                            color=OBSERVED_COLOR,
                            # symbol="circle" is default, so we don't need to specify
                        ),
                        text=[f"obs_{i}" for i in range(xyz_observed.shape[0])],
                        hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                        name=f"observed points (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                    )
                )
            else:
                # Add empty trace to maintain index consistency
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="markers",
                        marker=dict(size=5, color=OBSERVED_COLOR),
                        name=f"observed points (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                    )
                )
            
            # Correspondence lines from observed points to corresponding map points
            # Color by error: red if above threshold, green otherwise
            xyz_correspond = np.asarray(kfd["xyz_correspond"], dtype=float)
            residuals = np.asarray(kfd.get("residuals", np.zeros((0,), dtype=float)), dtype=float)
            
            if xyz_observed.size > 0 and xyz_correspond.size > 0 and xyz_observed.shape[0] == xyz_correspond.shape[0]:
                # Prettier colors: darker green and red
                GREEN_COLOR = "rgba(34,139,34,0.6)"  # Forest green
                RED_COLOR = "rgba(178,34,34,0.6)"    # Firebrick red
                
                # Create line segments for each correspondence with color based on error
                x_lines_green = []
                y_lines_green = []
                z_lines_green = []
                x_lines_red = []
                y_lines_red = []
                z_lines_red = []
                
                for i in range(xyz_observed.shape[0]):
                    if residuals.size > i and residuals[i] > ERROR_THRESHOLD:
                        # High error - red
                        x_lines_red.extend([xyz_observed[i, 0], xyz_correspond[i, 0], None])
                        y_lines_red.extend([xyz_observed[i, 1], xyz_correspond[i, 1], None])
                        z_lines_red.extend([xyz_observed[i, 2], xyz_correspond[i, 2], None])
                    else:
                        # Low error - green
                        x_lines_green.extend([xyz_observed[i, 0], xyz_correspond[i, 0], None])
                        y_lines_green.extend([xyz_observed[i, 1], xyz_correspond[i, 1], None])
                        z_lines_green.extend([xyz_observed[i, 2], xyz_correspond[i, 2], None])
                
                # Add green lines (low error)
                if len(x_lines_green) > 0:
                    fig.add_trace(
                        go.Scatter3d(
                            x=x_lines_green,
                            y=y_lines_green,
                            z=z_lines_green,
                            mode="lines",
                            line=dict(color=GREEN_COLOR, width=4),
                            name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                else:
                    # Add empty trace to maintain index consistency
                    fig.add_trace(
                        go.Scatter3d(
                            x=[],
                            y=[],
                            z=[],
                            mode="lines",
                            line=dict(color=GREEN_COLOR, width=4),
                            name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                
                # Add red lines (high error)
                if len(x_lines_red) > 0:
                    fig.add_trace(
                        go.Scatter3d(
                            x=x_lines_red,
                            y=y_lines_red,
                            z=z_lines_red,
                            mode="lines",
                            line=dict(color=RED_COLOR, width=4),
                            name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                else:
                    # Add empty trace to maintain index consistency
                    fig.add_trace(
                        go.Scatter3d(
                            x=[],
                            y=[],
                            z=[],
                            mode="lines",
                            line=dict(color=RED_COLOR, width=4),
                            name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
            else:
                # Add empty traces to maintain index consistency
                GREEN_COLOR = "rgba(34,139,34,0.6)"
                RED_COLOR = "rgba(178,34,34,0.6)"
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="lines",
                        line=dict(color=GREEN_COLOR, width=4),
                        name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                        showlegend=True,
                    )
                )
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="lines",
                        line=dict(color=RED_COLOR, width=4),
                        name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                        showlegend=True,
                    )
                )

        # Create slider steps
        # Each keyframe has 4 traces: map points, observed points, green correspondences, red correspondences
        # Plus 1 baseline trace
        steps = []
        for k, kfd in enumerate(keyframe_data):
            kf_idx = int(kfd.get("kf_idx", -1))
            frame_id = int(kfd.get("frame_id", -1))
            
            # Visibility: baseline always on, then 4 traces per keyframe
            vis = [True]  # baseline
            for i in range(len(keyframe_data)):
                # Map points
                vis.append(i == k)
                # Observed points
                vis.append(i == k)
                # Green correspondences (low error)
                vis.append(i == k)
                # Red correspondences (high error)
                vis.append(i == k)
            
            steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": vis},
                        {
                            "title": f"Map with observed points and correspondences (obj {OBJ_ID}) — kf_idx={kf_idx}, frame_id={frame_id}",
                        },
                    ],
                    label=str(frame_id if frame_id >= 0 else kf_idx),
                )
            )

        fig.update_layout(
            title=f"Map with observed points and correspondences (obj {OBJ_ID}) — kf_idx={int(keyframe_data[0].get('kf_idx', -1))}, frame_id={int(keyframe_data[0].get('frame_id', -1))}",
            margin=dict(l=0, r=0, b=0, t=55),
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            scene=scene_axes,
            sliders=[
                dict(
                    active=0,
                    currentvalue={"prefix": "frame_id: "},
                    steps=steps,
                )
            ],
            legend=dict(x=0.01, y=0.99),
            uirevision=f"kf_obs_corr_obj_{OBJ_ID}",
        )

        fig.show()

In [ ]:
# --- Plotly 3D slider with observed points, correspondences, AND newly sampled keypoints ---
if "global_landmarks_updates" not in globals() or not isinstance(global_landmarks_updates, list):
    print("No global_landmarks_updates found. Run the replay cell above first.")
elif len(global_landmarks_updates) == 0:
    print("global_landmarks_updates is empty. (Likely only the first keyframe ran, or the optimizer returned no landmarks.)")
else:
    updates = [u for u in global_landmarks_updates if int(u.get("obj_id", -1)) == int(OBJ_ID)]
    if len(updates) == 0:
        print(f"No landmark snapshots found for OBJ_ID={OBJ_ID}.")
    else:
        # Sort by keyframe index to make the slider monotonic
        updates = sorted(updates, key=lambda u: int(u.get("kf_idx", -1)))

        try:
            import plotly.graph_objects as go
        except Exception as e:
            raise ImportError(
                "Plotly is required for the interactive 3D plot. Install with `pip install plotly` and re-run this cell."
            ) from e

        # Import transform utilities
        from point2pose.utils.transform import transform_pts, inverse_SE3

        # Load observed points, correspondences, and newly sampled keypoints for each keyframe
        keyframe_data = []
        for kf_idx, row_idx in enumerate(kf_rows.tolist()):
            frame_id = int(frame_ids[row_idx])
            
            # Get object pose for this keyframe (to transform camera frame to object frame)
            # Follow the same pattern as notebook 12. frame_optimization_analysis.ipynb
            obj_pose = None
            if 'pose_frontend' in meta.files:
                obj_pose = meta['pose_frontend'][row_idx]
            elif 'pose_local' in meta.files:
                obj_pose = meta['pose_local'][row_idx]
            elif 'obj_pose' in meta.files:
                obj_pose = meta['obj_pose'][row_idx]
            elif 'obj_init_pose' in meta.files:
                obj_pose = meta['obj_init_pose'][row_idx]
            
            if obj_pose is not None:
                obj_pose = np.asarray(obj_pose, dtype=float)
                if obj_pose.shape == (4, 4):
                    # obj_pose transforms from object frame to camera frame
                    # We need inverse to transform camera frame to object frame
                    T_c2o = inverse_SE3(obj_pose)
                else:
                    T_c2o = None
            else:
                # If pose not available, skip transformation (will show incorrect positions)
                T_c2o = None
            
            
            # Get global optimized landmarks (map points)
            xyz_map = _ragged_slice_2d(meta, "obj_key_points", row_idx, 3, dtype=float)
            ids_map = np.arange(xyz_map.shape[0], dtype=int)
            
            # Filter valid keypoints
            if "obj_valid_data" in meta.files:
                valid = _ragged_slice(meta, "obj_valid", row_idx).astype(bool, copy=False).reshape(-1)
                if valid.size == xyz_map.shape[0]:
                    ids_map = np.flatnonzero(valid).astype(int)
                    xyz_map = xyz_map[valid]
            
            m = np.isfinite(xyz_map).all(axis=1)
            xyz_map = xyz_map[m]
            ids_map = ids_map[m]
            
            # Get newly sampled keypoints for this keyframe (keypoints added in this frame)
            xyz_newly_sampled = None
            ids_newly_sampled = None
            if "obj_key_point_frames_data" in meta.files:
                key_point_frames = _ragged_slice(meta, "obj_key_point_frames", row_idx).astype(int, copy=False)
                if key_point_frames.size == xyz_map.shape[0]:
                    # Find keypoints that were sampled in this frame
                    newly_sampled_mask = (key_point_frames == frame_id)
                    if np.any(newly_sampled_mask):
                        xyz_newly_sampled = xyz_map[newly_sampled_mask]
                        ids_newly_sampled = ids_map[newly_sampled_mask]
                    else:
                        xyz_newly_sampled = np.zeros((0, 3), dtype=float)
                        ids_newly_sampled = np.zeros((0,), dtype=int)
                else:
                    xyz_newly_sampled = np.zeros((0, 3), dtype=float)
                    ids_newly_sampled = np.zeros((0,), dtype=int)
            else:
                xyz_newly_sampled = np.zeros((0, 3), dtype=float)
                ids_newly_sampled = np.zeros((0,), dtype=int)
            
            # Get observed points (reg_curr3d) and corresponding map points (reg_key_points)
            # Note: reg_curr3d is in camera frame, reg_key_points is in object frame
            xyz_observed = None
            xyz_correspond = None
            if "reg_curr3d_data" in meta.files and "reg_key_points_data" in meta.files:
                xyz_observed_cam = _ragged_slice_2d(meta, "reg_curr3d", row_idx, 3, dtype=float)
                xyz_correspond = _ragged_slice_2d(meta, "reg_key_points", row_idx, 3, dtype=float)
                
                # Transform observed points from camera frame to object frame
                if T_c2o is not None and xyz_observed_cam.shape[0] > 0:
                    xyz_observed = transform_pts(T_c2o, xyz_observed_cam)
                else:
                    xyz_observed = xyz_observed_cam.copy() if xyz_observed_cam.shape[0] > 0 else np.zeros((0, 3), dtype=float)
                
                # Filter finite points - only if both arrays have the same number of points
                if xyz_observed.shape[0] == xyz_correspond.shape[0] and xyz_observed.shape[0] > 0:
                    m_obs = np.isfinite(xyz_observed).all(axis=1) & np.isfinite(xyz_correspond).all(axis=1)
                    if np.any(m_obs):
                        xyz_observed = xyz_observed[m_obs]
                        xyz_correspond = xyz_correspond[m_obs]
                    else:
                        xyz_observed = np.zeros((0, 3), dtype=float)
                        xyz_correspond = np.zeros((0, 3), dtype=float)
                else:
                    # Mismatched sizes or empty arrays - filter each separately but don't create correspondences
                    if xyz_observed.shape[0] > 0:
                        m_obs = np.isfinite(xyz_observed).all(axis=1)
                        xyz_observed = xyz_observed[m_obs] if np.any(m_obs) else np.zeros((0, 3), dtype=float)
                    else:
                        xyz_observed = np.zeros((0, 3), dtype=float)
                    # Set correspond to empty since sizes don't match
                    xyz_correspond = np.zeros((0, 3), dtype=float)
            
            # Get residuals for correspondence error coloring
            residuals = None
            if "reg_residuals_data" in meta.files and xyz_observed is not None and xyz_correspond is not None:
                residuals_flat = _ragged_slice(meta, "reg_residuals", row_idx)
                if residuals_flat.size > 0:
                    residuals = np.asarray(residuals_flat, dtype=float)
                    # Filter to match the filtered observed/correspond points
                    # We need to apply the same filtering that was applied to observed/correspond points
                    if "reg_curr3d_data" in meta.files and "reg_key_points_data" in meta.files:
                        xyz_observed_cam_raw = _ragged_slice_2d(meta, "reg_curr3d", row_idx, 3, dtype=float)
                        xyz_correspond_raw = _ragged_slice_2d(meta, "reg_key_points", row_idx, 3, dtype=float)
                        
                        if residuals.size == xyz_observed_cam_raw.shape[0] and xyz_observed_cam_raw.shape[0] == xyz_correspond_raw.shape[0]:
                            # Apply same filtering as observed points
                            m_obs = np.isfinite(xyz_observed_cam_raw).all(axis=1) & np.isfinite(xyz_correspond_raw).all(axis=1)
                            if np.any(m_obs):
                                residuals = residuals[m_obs]
                            else:
                                residuals = np.zeros((0,), dtype=float)
                        else:
                            residuals = np.zeros((0,), dtype=float)
                    else:
                        residuals = np.zeros((0,), dtype=float)
                else:
                    residuals = np.zeros((0,), dtype=float)
            else:
                residuals = np.zeros((0,), dtype=float)
            
            keyframe_data.append({
                "kf_idx": int(kf_idx),
                "frame_id": int(frame_id),
                "xyz_map": xyz_map,
                "ids_map": ids_map,
                "xyz_newly_sampled": xyz_newly_sampled if xyz_newly_sampled is not None else np.zeros((0, 3), dtype=float),
                "ids_newly_sampled": ids_newly_sampled if ids_newly_sampled is not None else np.zeros((0,), dtype=int),
                "xyz_observed": xyz_observed if xyz_observed is not None else np.zeros((0, 3), dtype=float),
                "xyz_correspond": xyz_correspond if xyz_correspond is not None else np.zeros((0, 3), dtype=float),
                "residuals": residuals if residuals is not None else np.zeros((0,), dtype=float),
            })

        # Establish axis ranges (no longer need color range since using fixed colors)
        _all_xyz = []
        for kfd in keyframe_data:
            xyz = kfd.get("xyz_map", None)
            if xyz is not None and xyz.size > 0:
                m = np.isfinite(xyz).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz[m])
            
            # Also include observed points and newly sampled points for axis range
            xyz_obs = kfd.get("xyz_observed", None)
            if xyz_obs is not None and xyz_obs.size > 0:
                m = np.isfinite(xyz_obs).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz_obs[m])
            
            xyz_new = kfd.get("xyz_newly_sampled", None)
            if xyz_new is not None and xyz_new.size > 0:
                m = np.isfinite(xyz_new).all(axis=1)
                if np.any(m):
                    _all_xyz.append(xyz_new[m])

        if len(_all_xyz) == 0:
            raise RuntimeError("No finite landmark xyz found in keyframe_data.")
        
        # Error threshold for correspondence coloring (adjust as needed)
        ERROR_THRESHOLD = 0.01  # meters

        all_xyz = np.concatenate(_all_xyz, axis=0)
        xmin, ymin, zmin = np.min(all_xyz, axis=0)
        xmax, ymax, zmax = np.max(all_xyz, axis=0)

        # Fixed axis ranges
        scene_axes = dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="z",
            aspectmode=ASPECT_MODE,
        )
        if FIX_AXIS_RANGES:
            spans = np.array([xmax - xmin, ymax - ymin, zmax - zmin], dtype=float)
            spans = np.maximum(spans, 1e-6)
            pad = 0.05 * float(np.max(spans))

            cx, cy, cz = float(0.5 * (xmin + xmax)), float(0.5 * (ymin + ymax)), float(0.5 * (zmin + zmax))
            if ASPECT_MODE == "cube":
                half = 0.5 * float(np.max(spans)) + pad
                xr = [cx - half, cx + half]
                yr = [cy - half, cy + half]
                zr = [cz - half, cz + half]
            else:
                xr = [float(xmin - pad), float(xmax + pad)]
                yr = [float(ymin - pad), float(ymax + pad)]
                zr = [float(zmin - pad), float(zmax + pad)]

            scene_axes.update(
                dict(
                    xaxis=dict(range=xr, autorange=False),
                    yaxis=dict(range=yr, autorange=False),
                    zaxis=dict(range=zr, autorange=False),
                )
            )

        # Baseline snapshot (first keyframe)
        base = keyframe_data[0]
        xyz0 = np.asarray(base["xyz_map"], dtype=float)
        ids0 = np.asarray(base["ids_map"], dtype=int).reshape(-1)
        m0 = np.isfinite(xyz0).all(axis=1)

        fig = go.Figure()

        # Baseline map (always visible) - same pretty blue color
        MAP_COLOR_BASELINE = "rgba(70,130,180,0.45)"  # Steel blue, more transparent for baseline
        fig.add_trace(
            go.Scatter3d(
                x=xyz0[m0, 0],
                y=xyz0[m0, 1],
                z=xyz0[m0, 2],
                mode="markers",
                marker=dict(size=2, color=MAP_COLOR_BASELINE),
                text=[f"id={int(i)}" for i in ids0[m0]],
                hovertemplate="%{text}<extra></extra>",
                name=f"baseline map (kf_idx={int(base.get('kf_idx', -1))}, frame_id={int(base.get('frame_id', -1))})",
            )
        )

        # For each keyframe, add:
        # 1. Map points (key points) for this keyframe
        # 2. Newly sampled keypoints for this keyframe
        # 3. Observed points
        # 4. Correspondence lines
        for k, kfd in enumerate(keyframe_data):
            kf_idx = int(kfd.get("kf_idx", -1))
            frame_id = int(kfd.get("frame_id", -1))
            
            # Map points (key points) for this keyframe - pretty blue color
            xyz_map = np.asarray(kfd["xyz_map"], dtype=float)
            ids_map = np.asarray(kfd["ids_map"], dtype=int).reshape(-1)
            m_map = np.isfinite(xyz_map).all(axis=1)
            
            # Pretty blue color for map points
            MAP_COLOR = "rgba(70,130,180,0.85)"  # Steel blue
            
            fig.add_trace(
                go.Scatter3d(
                    x=xyz_map[m_map, 0],
                    y=xyz_map[m_map, 1],
                    z=xyz_map[m_map, 2],
                    mode="markers",
                    marker=dict(
                        size=4,
                        color=MAP_COLOR,
                        opacity=0.9,
                    ),
                    text=[f"id={int(i)}, frame={frame_id}" for i in ids_map[m_map]],
                    hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                    name=f"key points (kf_idx={kf_idx}, frame_id={frame_id})",
                    visible=(k == 0),
                )
            )
            
            # Newly sampled keypoints for this keyframe - pretty purple/magenta color
            xyz_newly_sampled = np.asarray(kfd["xyz_newly_sampled"], dtype=float)
            ids_newly_sampled = np.asarray(kfd["ids_newly_sampled"], dtype=int).reshape(-1)
            
            # Pretty purple/magenta color for newly sampled keypoints
            NEWLY_SAMPLED_COLOR = "rgba(186,85,211,0.9)"  # Medium orchid
            
            if xyz_newly_sampled.size > 0:
                m_new = np.isfinite(xyz_newly_sampled).all(axis=1)
                if np.any(m_new):
                    fig.add_trace(
                        go.Scatter3d(
                            x=xyz_newly_sampled[m_new, 0],
                            y=xyz_newly_sampled[m_new, 1],
                            z=xyz_newly_sampled[m_new, 2],
                            mode="markers",
                            marker=dict(
                                size=6,
                                color=NEWLY_SAMPLED_COLOR,
                                # symbol="circle" is default, so we don't need to specify
                            ),
                            text=[f"new_kp_id={int(i)}, frame={frame_id}" for i in ids_newly_sampled[m_new]],
                            hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                            name=f"newly sampled keypoints (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                        )
                    )
                else:
                    # Add empty trace to maintain index consistency
                    fig.add_trace(
                        go.Scatter3d(
                            x=[],
                            y=[],
                            z=[],
                            mode="markers",
                            marker=dict(size=6, color=NEWLY_SAMPLED_COLOR),
                            name=f"newly sampled keypoints (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                        )
                    )
            else:
                # Add empty trace to maintain index consistency
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="markers",
                        marker=dict(size=6, color=NEWLY_SAMPLED_COLOR),
                        name=f"newly sampled keypoints (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                    )
                )
            
            # Observed points (already filtered when loaded) - pretty coral/salmon color
            xyz_observed = np.asarray(kfd["xyz_observed"], dtype=float)
            
            # Pretty coral/salmon color for observed points
            OBSERVED_COLOR = "rgba(255,127,80,0.85)"  # Coral
            
            if xyz_observed.size > 0:
                fig.add_trace(
                    go.Scatter3d(
                        x=xyz_observed[:, 0],
                        y=xyz_observed[:, 1],
                        z=xyz_observed[:, 2],
                        mode="markers",
                        marker=dict(
                            size=5,
                            color=OBSERVED_COLOR,
                            # symbol="circle" is default, so we don't need to specify
                        ),
                        text=[f"obs_{i}" for i in range(xyz_observed.shape[0])],
                        hovertemplate="x=%{x:.4f}<br>y=%{y:.4f}<br>z=%{z:.4f}<br>%{text}<extra></extra>",
                        name=f"observed points (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                    )
                )
            else:
                # Add empty trace to maintain index consistency
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="markers",
                        marker=dict(size=5, color=OBSERVED_COLOR),
                        name=f"observed points (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                    )
                )
            
            # Correspondence lines from observed points to corresponding map points
            # Color by error: red if above threshold, green otherwise
            xyz_correspond = np.asarray(kfd["xyz_correspond"], dtype=float)
            residuals = np.asarray(kfd.get("residuals", np.zeros((0,), dtype=float)), dtype=float)
            
            if xyz_observed.size > 0 and xyz_correspond.size > 0 and xyz_observed.shape[0] == xyz_correspond.shape[0]:
                # Prettier colors: darker green and red
                GREEN_COLOR = "rgba(34,139,34,0.6)"  # Forest green
                RED_COLOR = "rgba(178,34,34,0.6)"    # Firebrick red
                
                # Create line segments for each correspondence with color based on error
                x_lines_green = []
                y_lines_green = []
                z_lines_green = []
                x_lines_red = []
                y_lines_red = []
                z_lines_red = []
                
                for i in range(xyz_observed.shape[0]):
                    if residuals.size > i and residuals[i] > ERROR_THRESHOLD:
                        # High error - red
                        x_lines_red.extend([xyz_observed[i, 0], xyz_correspond[i, 0], None])
                        y_lines_red.extend([xyz_observed[i, 1], xyz_correspond[i, 1], None])
                        z_lines_red.extend([xyz_observed[i, 2], xyz_correspond[i, 2], None])
                    else:
                        # Low error - green
                        x_lines_green.extend([xyz_observed[i, 0], xyz_correspond[i, 0], None])
                        y_lines_green.extend([xyz_observed[i, 1], xyz_correspond[i, 1], None])
                        z_lines_green.extend([xyz_observed[i, 2], xyz_correspond[i, 2], None])
                
                # Add green lines (low error)
                if len(x_lines_green) > 0:
                    fig.add_trace(
                        go.Scatter3d(
                            x=x_lines_green,
                            y=y_lines_green,
                            z=z_lines_green,
                            mode="lines",
                            line=dict(color=GREEN_COLOR, width=4),
                            name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                else:
                    # Add empty trace to maintain index consistency
                    fig.add_trace(
                        go.Scatter3d(
                            x=[],
                            y=[],
                            z=[],
                            mode="lines",
                            line=dict(color=GREEN_COLOR, width=4),
                            name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                
                # Add red lines (high error)
                if len(x_lines_red) > 0:
                    fig.add_trace(
                        go.Scatter3d(
                            x=x_lines_red,
                            y=y_lines_red,
                            z=z_lines_red,
                            mode="lines",
                            line=dict(color=RED_COLOR, width=4),
                            name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
                else:
                    # Add empty trace to maintain index consistency
                    fig.add_trace(
                        go.Scatter3d(
                            x=[],
                            y=[],
                            z=[],
                            mode="lines",
                            line=dict(color=RED_COLOR, width=4),
                            name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                            visible=(k == 0),
                            showlegend=True,
                        )
                    )
            else:
                # Add empty traces to maintain index consistency
                GREEN_COLOR = "rgba(34,139,34,0.6)"
                RED_COLOR = "rgba(178,34,34,0.6)"
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="lines",
                        line=dict(color=GREEN_COLOR, width=4),
                        name=f"correspondences (low error) (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                        showlegend=True,
                    )
                )
                fig.add_trace(
                    go.Scatter3d(
                        x=[],
                        y=[],
                        z=[],
                        mode="lines",
                        line=dict(color=RED_COLOR, width=4),
                        name=f"correspondences (high error) (kf_idx={kf_idx}, frame_id={frame_id})",
                        visible=(k == 0),
                        showlegend=True,
                    )
                )

        # Create slider steps
        # Each keyframe has 5 traces: map points, newly sampled keypoints, observed points, green correspondences, red correspondences
        # Plus 1 baseline trace
        steps = []
        for k, kfd in enumerate(keyframe_data):
            kf_idx = int(kfd.get("kf_idx", -1))
            frame_id = int(kfd.get("frame_id", -1))
            
            # Visibility: baseline always on, then 5 traces per keyframe
            vis = [True]  # baseline
            for i in range(len(keyframe_data)):
                # Map points
                vis.append(i == k)
                # Newly sampled keypoints
                vis.append(i == k)
                # Observed points
                vis.append(i == k)
                # Green correspondences (low error)
                vis.append(i == k)
                # Red correspondences (high error)
                vis.append(i == k)
            
            steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": vis},
                        {
                            "title": f"Map with observed points, correspondences, and newly sampled keypoints (obj {OBJ_ID}) — kf_idx={kf_idx}, frame_id={frame_id}",
                        },
                    ],
                    label=str(frame_id if frame_id >= 0 else kf_idx),
                )
            )

        fig.update_layout(
            title=f"Map with observed points, correspondences, and newly sampled keypoints (obj {OBJ_ID}) — kf_idx={int(keyframe_data[0].get('kf_idx', -1))}, frame_id={int(keyframe_data[0].get('frame_id', -1))}",
            margin=dict(l=0, r=0, b=0, t=55),
            width=PLOT_WIDTH,
            height=PLOT_HEIGHT,
            scene=scene_axes,
            sliders=[
                dict(
                    active=0,
                    currentvalue={"prefix": "frame_id: "},
                    steps=steps,
                )
            ],
            legend=dict(x=0.01, y=0.99),
            uirevision=f"kf_obs_corr_newkp_obj_{OBJ_ID}",
        )

        fig.show()